# 2B core benchmark

Full-budget comparison of the seed-42 and seed-43 core trios, plus the selected seed-42 additive-SatCLIP extension and matched shuffled-coordinate controls.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs" / "evaluation"
runs = {
    "no_loc": "11437",
    "loc_text integer": "11441",
    "loc_embed L40": "11438",
    "additive SatCLIP": "11619",
    "no_loc (seed 43)": "11622",
    "loc_text integer (seed 43)": "11624",
    "loc_embed L40 (seed 43)": "11627",
}
shuffled_runs = {
    "loc_text integer": "11445",
    "loc_embed L40": "11446",
    "additive SatCLIP": "11620",
    "loc_text integer (seed 43)": "11625",
    "loc_embed L40 (seed 43)": "11628",
}
seed_by_condition = {condition: (43 if "seed 43" in condition else 42) for condition in runs}
condition_order = list(runs)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in runs.items()
}

predictions = {
    condition: pd.read_json(evaluation_root / job / "predictions.jsonl", lines=True)
    for condition, job in runs.items()
}

sample_scores = {
    condition: pd.read_json(
        evaluation_root / job / "scored_predictions" / "sample_scores.jsonl",
        lines=True,
    )
    for condition, job in runs.items()
}

pd.DataFrame({
    "Condition": condition_order,
    "Evaluation job": [runs[c] for c in condition_order],
    "Samples": [len(predictions[c]) for c in condition_order],
})

## Shuffled-coordinate controls

Values are shuffled minus correct. Negative values mean that replacing the true coordinates hurt performance.

In [ ]:
shuffled_summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in shuffled_runs.items()
}

def counterfactual_task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

def primary_metrics(summary):
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": counterfactual_task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": counterfactual_task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": counterfactual_task_row(summary, "bounding box")["miou"],
    }

counterfactual_rows = []
for condition in shuffled_runs:
    correct = primary_metrics(summaries[condition])
    shuffled = primary_metrics(shuffled_summaries[condition])
    counterfactual_rows.append({
        "Condition": condition,
        **{metric: shuffled[metric] - correct[metric] for metric in correct},
    })

counterfactual_deltas = pd.DataFrame(counterfactual_rows).set_index("Condition")
counterfactual_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-counterfactual_deltas.abs().to_numpy().max(), vmax=counterfactual_deltas.abs().to_numpy().max()).set_caption("Shuffled − correct coordinates")

### Direct-geography MCQs under shuffling

In [ ]:
def category_accuracy(summary, category):
    return next(
        row["accuracy"]
        for row in summary["by_task_category"]
        if row["task_type"] == "mcq" and row["task_category"] == category
    )

geo_shuffle_rows = []
for condition in shuffled_runs:
    for category in ["country", "climate zone", "season"]:
        correct = category_accuracy(summaries[condition], category)
        shuffled = category_accuracy(shuffled_summaries[condition], category)
        geo_shuffle_rows.append({
            "Condition": condition,
            "Category": category,
            "Correct": correct,
            "Shuffled": shuffled,
            "Difference": shuffled - correct,
        })

pd.DataFrame(geo_shuffle_rows).style.format({"Correct": "{:.3f}", "Shuffled": "{:.3f}", "Difference": "{:+.3f}"})

## Population check

In [ ]:
id_sets = {condition: set(frame["sample_id"].astype(str)) for condition, frame in predictions.items()}
reference_ids = id_sets[condition_order[0]]

population_check = pd.DataFrame([
    {
        "Condition": condition,
        "Seed": seed_by_condition[condition],
        "Rows": len(predictions[condition]),
        "Unique sample IDs": predictions[condition]["sample_id"].astype(str).nunique(),
        "Same IDs as no_loc": ids == reference_ids,
    }
    for condition, ids in id_sets.items()
])
population_check

## Main results

One primary metric per task family. Average rank weights the four task families equally; lower is better.

In [ ]:
def task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

main_results = pd.DataFrame([
    {
        "Condition": condition,
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": task_row(summary, "bounding box")["miou"],
    }
    for condition, summary in summaries.items()
]).set_index("Condition").reindex(condition_order)

metric_columns = ["Caption BLEU-4", "Binary accuracy", "MCQ accuracy", "Bounding-box mIoU"]
main_results["Average rank"] = main_results[metric_columns].rank(ascending=False).mean(axis=1)
main_results.style.format({"Seed": "{:.0f}", **{column: "{:.4f}" for column in [*metric_columns, "Average rank"]}}).highlight_max(
    subset=metric_columns, axis=0, props="font-weight: bold"
).highlight_min(
    subset=["Average rank"], axis=0, props="font-weight: bold"
).set_caption("Primary benchmark metrics")

## Difference from no_loc

Positive values favor the location-conditioned model.

In [ ]:
delta_pairs = [
    ("loc_text integer, seed 42", "loc_text integer", "no_loc"),
    ("loc_embed L40, seed 42", "loc_embed L40", "no_loc"),
    ("additive SatCLIP, seed 42", "additive SatCLIP", "no_loc"),
    ("loc_text integer, seed 43", "loc_text integer (seed 43)", "no_loc (seed 43)"),
    ("loc_embed L40, seed 43", "loc_embed L40 (seed 43)", "no_loc (seed 43)"),
]
delta = pd.DataFrame({label: main_results.loc[condition, metric_columns] - main_results.loc[baseline, metric_columns] for label, condition, baseline in delta_pairs}).T
limit = delta.abs().to_numpy().max()
delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Location condition − no_loc")

## Seed replication

Seed-43 minus seed-42 values show training-seed variation for the core trio. Two-seed means are descriptive, not confidence intervals.

In [ ]:
replicated_conditions = {
    "no_loc": ("no_loc", "no_loc (seed 43)"),
    "loc_text integer": ("loc_text integer", "loc_text integer (seed 43)"),
    "loc_embed L40": ("loc_embed L40", "loc_embed L40 (seed 43)"),
}
seed_differences = pd.DataFrame({
    condition: main_results.loc[seed43, metric_columns] - main_results.loc[seed42, metric_columns]
    for condition, (seed42, seed43) in replicated_conditions.items()
}).T
two_seed_means = pd.DataFrame({
    condition: main_results.loc[[seed42, seed43], metric_columns].mean()
    for condition, (seed42, seed43) in replicated_conditions.items()
}).T
display(seed_differences.style.format("{:+.4f}").set_caption("Seed 43 minus seed 42"))
display(two_seed_means.style.format("{:.4f}").set_caption("Two-seed mean"))

## Task-wise results

In [ ]:
category_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_category"]:
        category_rows.append({"Condition": condition, **row})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    rows = category_scores[category_scores["task_type"] == task_type]
    table = rows.pivot(index="Condition", columns="task_category", values=metric)
    overall = pd.Series({
        condition: task_row(summaries[condition], task_type)[metric]
        for condition in condition_order
    }, name="Overall")
    return table.reindex(condition_order).join(overall)

display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy"))
display(category_table("mcq", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU"))

## Geography-sensitive MCQs

Country, climate-zone and season questions are the clearest direct test of whether the location tokens carry useful geographic information.

In [ ]:
geo_categories = ["country", "climate zone", "season"]
geo_mcq = (
    category_scores[
        (category_scores["task_type"] == "mcq")
        & category_scores["task_category"].isin(geo_categories)
    ]
    .pivot(index="Condition", columns="task_category", values="accuracy")
    .reindex(condition_order)
)
geo_mcq["Mean"] = geo_mcq.mean(axis=1)
geo_mcq.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Direct-geography MCQ accuracy")

## Paired changes: seed 42

Counts below show whether seed-42 `loc_embed` fixes or breaks the exact same binary and MCQ samples relative to the seed-42 baseline.

In [ ]:
base = sample_scores["no_loc"][["sample_id", "task_type", "task_category", "correct"]].rename(columns={"correct": "no_loc_correct"})
embed = sample_scores["loc_embed L40"][["sample_id", "correct"]].rename(columns={"correct": "loc_embed_correct"})
paired = base.merge(embed, on="sample_id", validate="one_to_one")
paired = paired[paired["task_type"].isin(["binary", "mcq"])].copy()

def transition(row):
    if row.no_loc_correct and not row.loc_embed_correct:
        return "Broken by loc_embed"
    if not row.no_loc_correct and row.loc_embed_correct:
        return "Fixed by loc_embed"
    if row.no_loc_correct:
        return "Both correct"
    return "Both wrong"

paired["Transition"] = paired.apply(transition, axis=1)
transition_table = pd.crosstab(
    [paired["task_type"], paired["task_category"]],
    paired["Transition"],
).fillna(0).astype(int)
transition_table

## Qualitative changes: seed 42

Examples where the two models produced different answers. Change `task_filter` and `category_filter` to inspect a weakness or a location-sensitive subtask.

In [ ]:
task_filter = "mcq"
category_filter = "country"
number_of_examples = 12

keep = ["sample_id", "input_text", "target_texts", "prediction", "task_type", "task_category", "country", "lat", "lon"]
base_predictions = predictions["no_loc"][keep].rename(columns={"prediction": "no_loc prediction"})
embed_predictions = predictions["loc_embed L40"][["sample_id", "prediction"]].rename(columns={"prediction": "loc_embed prediction"})
comparison = base_predictions.merge(embed_predictions, on="sample_id", validate="one_to_one")
changed = comparison[
    (comparison["task_type"] == task_filter)
    & (comparison["task_category"] == category_filter)
    & (comparison["no_loc prediction"] != comparison["loc_embed prediction"])
]
changed.head(number_of_examples)

## Criterion-selected caption examples

Per-caption BLEU-4, METEOR, CIDEr and ROUGE-L changes are computed against `no_loc` from the same seed. The displayed cases are the two strongest metric-disagreement cases per location condition, ranked by the sum of absolute within-metric percentile changes. This avoids selecting examples merely because they favor one method.

In [ ]:
from nltk.translate.meteor_score import meteor_score
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def caption_sample_metrics(frame):
    captions = frame[frame["task_type"] == "captioning"].reset_index(drop=True)
    references = {i: list(row.target_texts) for i, row in captions.iterrows()}
    candidates = {i: [row.prediction] for i, row in captions.iterrows()}
    _, bleu = Bleu(4).compute_score(references, candidates)
    _, cider = Cider().compute_score(references, candidates)
    metrics = []
    for i, row in captions.iterrows():
        reference = row.target_texts[0]
        metrics.append({
            "sample_id": str(row.sample_id),
            "BLEU-4": float(bleu[3][i]),
            "METEOR": float(meteor_score([reference.split()], row.prediction.split())),
            "CIDEr": float(cider[i]),
            "ROUGE-L": float(rouge.score(reference, row.prediction)["rougeL"].fmeasure),
        })
    return pd.DataFrame(metrics)

caption_metrics = {condition: caption_sample_metrics(frame) for condition, frame in predictions.items()}

In [ ]:
caption_comparisons = [
    ("loc_text integer, seed 42", "loc_text integer", "no_loc"),
    ("loc_embed L40, seed 42", "loc_embed L40", "no_loc"),
    ("loc_text integer, seed 43", "loc_text integer (seed 43)", "no_loc (seed 43)"),
    ("loc_embed L40, seed 43", "loc_embed L40 (seed 43)", "no_loc (seed 43)"),
]
caption_delta_rows = []
metric_names = ["BLEU-4", "METEOR", "CIDEr", "ROUGE-L"]
for label, condition, baseline in caption_comparisons:
    paired_metrics = caption_metrics[condition].merge(
        caption_metrics[baseline], on="sample_id", suffixes=("_condition", "_baseline"), validate="one_to_one"
    )
    deltas = pd.DataFrame({metric: paired_metrics[f"{metric}_condition"] - paired_metrics[f"{metric}_baseline"] for metric in metric_names})
    disagreement = (deltas.min(axis=1) < 0) & (deltas.max(axis=1) > 0)
    severity = deltas.abs().rank(pct=True).sum(axis=1)
    condition_rows = predictions[condition].set_index(predictions[condition]["sample_id"].astype(str))
    baseline_rows = predictions[baseline].set_index(predictions[baseline]["sample_id"].astype(str))
    for index in paired_metrics.index[disagreement]:
        sample_id = str(paired_metrics.loc[index, "sample_id"])
        row = condition_rows.loc[sample_id]
        caption_delta_rows.append({
            "Comparison": label, "sample_id": sample_id, "Country": row.country,
            "Reference": row.target_texts[0], "no_loc prediction": baseline_rows.loc[sample_id].prediction,
            "location prediction": row.prediction, "Selection score": severity[index],
            **{f"Δ {metric}": deltas.loc[index, metric] for metric in metric_names},
        })
caption_disagreements = pd.DataFrame(caption_delta_rows)
selected_caption_examples = (
    caption_disagreements.sort_values(["Comparison", "Selection score"], ascending=[True, False])
    .groupby("Comparison", sort=False).head(2).reset_index(drop=True)
)
display(caption_disagreements.groupby("Comparison").size().rename("Metric-disagreement captions").to_frame())
selected_caption_examples

## Full diagnostic tables

These retain sample counts, extraction rates, secondary caption metrics and bounding-box thresholds.

In [ ]:
task_type_rows = []
caption_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_type"]:
        task_type_rows.append({"Condition": condition, **row})
    caption_rows.append({"Condition": condition, **summary["captioning"]})

display(pd.DataFrame(task_type_rows).sort_values(["task_type", "Condition"]).reset_index(drop=True))
display(pd.DataFrame(caption_rows).set_index("Condition").reindex(condition_order).reset_index())
display(category_scores.sort_values(["task_type", "task_category", "Condition"]).reset_index(drop=True))

## Current reading

- The seed-43 core trio closely reproduces the seed-42 pattern: location conditions slightly reduce binary accuracy, while `loc_embed` gives the larger MCQ and BLEU-4 gain.
- Direct-geography gains and shuffled-coordinate damage replicate across seeds; coordinate use is much larger and more stable than the small aggregate advantage over `no_loc`.
- Caption ordering is not stable across metrics or seeds: seed-42 `loc_embed` loses CIDEr to `no_loc`, while seed 43 exceeds it.
- The full additive-SatCLIP extension closely matches token-based `loc_embed` and becomes strongly coordinate-sensitive, unlike its seed-43 1000-step run.
- No condition dominates every task family at full budget.